In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

# DONT SHARE THIS FILE


# NO figures download

In [ ]:
"""
Elsevier Full-Text API Fetcher
===============================
Reads download_failed.txt, fetches full text for all Elsevier DOIs
via the Elsevier TDM (Text and Data Mining) API.

Requirements:
    - Must be on K-State network or K-State VPN for full-text access
    - pip install requests

Output:
    - Full text saved as .txt files in TXT_OUTPUT folder
    - These feed directly into your preprocessing notebook
      (skips Azure OCR entirely for Elsevier papers)
    - download_failed_remaining.txt lists any DOIs still not retrieved

Usage:
    1. Set ELSEVIER_API_KEY in your .env file
    2. Make sure download_failed.txt exists (produced by pdf_downloader.py)
    3. Run: python elsevier_api_fetch.py
"""

import os
import time
import requests

# ── CONFIGURATION ─────────────────────────────────────────────────────────────

API_KEY        = os.getenv("ELSEVIER_API_KEY")
FAILED_LOG     = "ClosedAccess_failed.txt" # written by pdf_downloader.py
TXT_OUTPUT     = "Elsevier_TXT_files"           # folder for retrieved full texts
STILL_FAILED   = "download_failed_remaining.txt"

# ── HELPERS ───────────────────────────────────────────────────────────────────

def load_failed_dois(filepath):
    """Read all DOIs from download_failed.txt."""
    if not os.path.exists(filepath):
        print(f"[ERROR] {filepath} not found.")
        print("        Run pdf_downloader.py first to generate this file.")
        return []
    with open(filepath, "r") as f:
        dois = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(dois)} failed DOIs from {filepath}")
    return dois


def is_elsevier(doi):
    """
    Elsevier DOIs start with 10.1016 (ScienceDirect journals).
    Other Elsevier prefixes are included below as well.
    """
    elsevier_prefixes = [
        "10.1016",   # ScienceDirect (main)
        "10.1006",   # older Academic Press journals
        "10.1053",   # some clinical journals
        "10.1054",
        "10.1067",
    ]
    return any(doi.startswith(p) for p in elsevier_prefixes)


def already_fetched(doi, output_folder):
    safe_name = doi.replace("/", "_").replace(".", "-") + ".txt"
    return os.path.exists(os.path.join(output_folder, safe_name))


def fetch_elsevier_fulltext(doi, api_key, output_folder):
    """
    Fetch full text for one Elsevier DOI via the TDM API.

    Returns:
        "full"     -- full article text retrieved and saved
        "abstract" -- only abstract returned (not on K-State network)
        "failed"   -- API error or no content
    """
    url     = f"https://api.elsevier.com/content/article/doi/{doi}"
    headers = {
        "X-ELS-APIKey" : api_key,
        "Accept"       : "text/plain",   # clean UTF-8 text, no XML markup
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)

        if response.status_code == 200:
            text      = response.text
            word_count = len(text.split())

            # A full article is typically 3000+ words
            # An abstract-only response is usually under 500 words
            if word_count < 500:
                print(f"  [ABSTRACT ONLY] {doi} ({word_count} words)")
                print(f"  You may not be on K-State network/VPN.")
                return "abstract"

            # Save full text
            safe_name = doi.replace("/", "_").replace(".", "-") + ".txt"
            out_path  = os.path.join(output_folder, safe_name)
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(text)
            print(f"  [OK] {doi} ({word_count} words)")
            return "full"

        elif response.status_code == 401:
            print(f"  [AUTH ERROR] Invalid API key for: {doi}")
            return "failed"

        elif response.status_code == 403:
            print(f"  [ACCESS DENIED] No entitlement for: {doi}")
            print(f"  Check K-State network connection or VPN.")
            return "failed"

        elif response.status_code == 404:
            print(f"  [NOT FOUND] DOI not in Elsevier API: {doi}")
            return "failed"

        else:
            print(f"  [ERROR] Status {response.status_code} for: {doi}")
            print(f"  Response: {response.text[:200]}")
            return "failed"

    except Exception as e:
        print(f"  [EXCEPTION] {doi}: {e}")
        return "failed"


# ── MAIN ──────────────────────────────────────────────────────────────────────

def run():
    os.makedirs(TXT_OUTPUT, exist_ok=True)

    all_failed   = load_failed_dois(FAILED_LOG)
    elsevier     = [d for d in all_failed if is_elsevier(d)]
    non_elsevier = [d for d in all_failed if not is_elsevier(d)]

    print(f"\nElsevier DOIs to fetch : {len(elsevier)}")
    print(f"Non-Elsevier (skipped) : {len(non_elsevier)}")
    print(f"Output folder          : {TXT_OUTPUT}/\n")

    if not elsevier:
        print("No Elsevier DOIs found in failed list.")
        return

    # Test the API key with the first DOI before running all
    print("=== Testing API key with first DOI ===")
    test_result = fetch_elsevier_fulltext(elsevier[0], API_KEY, TXT_OUTPUT)
    if test_result == "failed":
        print("\n[STOP] API key test failed. Check your key and try again.")
        return
    if test_result == "abstract":
        print("\n[WARNING] Only getting abstracts. Connect to K-State VPN and rerun.")
        return

    print("\n=== Fetching all Elsevier DOIs ===")
    results = {"full": [], "abstract": [], "failed": []}

    for doi in elsevier:
        if already_fetched(doi, TXT_OUTPUT):
            print(f"  [SKIP] Already fetched: {doi}")
            results["full"].append(doi)
            continue
        result = fetch_elsevier_fulltext(doi, API_KEY, TXT_OUTPUT)
        results[result].append(doi)
        time.sleep(0.5)   # 2 requests/second -- well within API rate limits

    # Write remaining failures
    still_failed = results["abstract"] + results["failed"] + non_elsevier
    with open(STILL_FAILED, "w") as f:
        f.write("\n".join(still_failed))

    print(f"\n{'='*60}")
    print(f"=== ELSEVIER API SUMMARY ===")
    print(f"{'='*60}")
    print(f"  Full text retrieved : {len(results['full'])}")
    print(f"  Abstract only       : {len(results['abstract'])}")
    print(f"  Failed              : {len(results['failed'])}")
    print(f"\n  Non-Elsevier DOIs still needing manual action ({len(non_elsevier)}):")
    for doi in non_elsevier:
        print(f"    {doi}")
    print(f"\n  Remaining failures saved to: {STILL_FAILED}")
    print(f"  Options: ILL at lib.k-state.edu or email corresponding author")
    print(f"{'='*60}")
    print(f"\nNOTE: TXT files in {TXT_OUTPUT}/ feed directly into your")
    print(f"preprocessing notebook, bypassing Azure OCR entirely.")


if __name__ == "__main__":
    run()

In [ ]:
"""
Elsevier Full-Text XML Fetcher
================================
Reads download_failed.txt, fetches full-text XML for all Elsevier DOIs,
parses into clean structured text, and downloads all figure images.

Requirements:
    - Must be on K-State network or K-State VPN for full-text access
    - pip install requests lxml

Output:
    Elsevier_TXT_files/   -- clean structured text, feeds into preprocessing notebook
    Elsevier_FIG_files/   -- figure images, can be sent to vision model for data extraction
    download_failed_elsevier.txt -- DOIs still not retrieved
"""

import os
import re
import time
import requests
from lxml import etree

# ── CONFIGURATION ─────────────────────────────────────────────────────────────

API_KEY      = os.getenv("ELSEVIER_API_KEY")
FAILED_LOG   = "doi_files/Relevant_DOI_250_ARTICLES.txt" # written by pdf_downloader.py
TXT_OUTPUT   = "Elsevier_TXT_files"
FIG_OUTPUT   = "Elsevier_FIG_files"
STILL_FAILED = "download_failed_elsevier.txt"

# Elsevier XML uses these namespace URIs
CE   = "http://www.elsevier.com/xml/common/dtd"
XOCS = "http://www.elsevier.com/xml/xocs/dtd"
JA   = "http://www.elsevier.com/xml/ja/dtd"
DC   = "http://purl.org/dc/elements/1.1/"
XLINK = "http://www.w3.org/1999/xlink"
TB   = "http://www.elsevier.com/xml/common/table/dtd"

# ── HELPERS ───────────────────────────────────────────────────────────────────

def load_failed_dois(filepath):
    if not os.path.exists(filepath):
        print(f"[ERROR] {filepath} not found.")
        return []
    with open(filepath, "r") as f:
        dois = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(dois)} failed DOIs from {filepath}")
    return dois


def is_elsevier(doi):
    return any(doi.startswith(p) for p in ["10.1016","10.1006","10.1053","10.1054"])


def safe_filename(doi, suffix=""):
    return doi.replace("/", "_").replace(".", "-") + suffix


def already_fetched(doi):
    return os.path.exists(os.path.join(TXT_OUTPUT, safe_filename(doi, ".txt")))


# ── TEXT EXTRACTION ───────────────────────────────────────────────────────────

def get_all_text(node):
    """
    Recursively extract all text from an XML node, stripping tags.
    Handles mixed content (text between child elements).
    """
    parts = []
    if node.text and node.text.strip():
        parts.append(node.text.strip())
    for child in node:
        child_text = get_all_text(child)
        if child_text:
            parts.append(child_text)
        if child.tail and child.tail.strip():
            parts.append(child.tail.strip())
    return " ".join(p for p in parts if p)


# ── TABLE PARSER ──────────────────────────────────────────────────────────────

def parse_ce_table(table_node):
    """
    Parse an Elsevier ce:table element into readable text.
    Elsevier tables use ce:tgroup > ce:thead/ce:tbody > ce:row > ce:entry.
    Also handles the tb: namespace variant.
    """
    lines = []

    # Try to get table label and caption
    label   = table_node.find(f"{{{CE}}}label")
    caption = table_node.find(f".//{{{CE}}}caption")
    if label is not None and label.text:
        lines.append(f"TABLE {label.text.strip()}")
    if caption is not None:
        cap_text = get_all_text(caption)
        if cap_text:
            lines.append(f"Caption: {cap_text}")

    # Find all rows -- try both ce: and tb: namespaces
    all_rows = (
        table_node.findall(f".//{{{CE}}}row") or
        table_node.findall(f".//{{{TB}}}row") or
        table_node.findall(".//{*}row")
    )

    header_done = False
    for row in all_rows:
        entries = (
            row.findall(f"{{{CE}}}entry") or
            row.findall(f"{{{TB}}}entry") or
            row.findall("{*}entry")
        )
        if not entries:
            continue
        cell_texts = []
        for entry in entries:
            cell_text = get_all_text(entry)
            cell_texts.append(cell_text if cell_text else "")
        row_str = " | ".join(cell_texts)
        if row_str.strip():
            lines.append(row_str)
            # Add separator after header row
            if not header_done:
                lines.append("-" * len(row_str))
                header_done = True

    return "\n".join(lines) if lines else ""


# ── FIGURE URL EXTRACTOR ──────────────────────────────────────────────────────

def extract_figure_urls(root, xml_bytes):
    """
    Extract figure image URLs from Elsevier XML.

    Elsevier stores figure URLs in two ways:
    1. <ce:link xlink:href="..."> inside <ce:figure>
    2. Direct API URLs embedded in the raw XML like:
       api.elsevier.com/content/object/eid/1-s2.0-XXXX-gr1.jpg

    We try both approaches and combine results.
    """
    figures = []

    # Method 1: parse ce:figure elements
    for fig in root.findall(f".//{{{CE}}}figure"):
        # Get caption
        caption_node = fig.find(f".//{{{CE}}}caption")
        caption = get_all_text(caption_node) if caption_node is not None else ""

        # Get image URL from ce:link xlink:href
        link = fig.find(f".//{{{CE}}}link")
        href = None
        if link is not None:
            href = link.get(f"{{{XLINK}}}href") or link.get("href") or link.text

        # Also try direct href on the figure element itself
        if not href:
            href = fig.get(f"{{{XLINK}}}href") or fig.get("href")

        if href and ("http" in href or href.startswith("//")):
            if not href.startswith("http"):
                href = "https:" + href
            figures.append((caption, href))

    # Method 2: scan raw XML bytes for Elsevier object API URLs
    # These look like: api.elsevier.com/content/object/eid/1-s2.0-XXX-gr1.jpg
    if not figures:
        raw_text = xml_bytes.decode("utf-8", errors="replace")
        pattern  = r'https://api\.elsevier\.com/content/object/eid/[^\s<>"\'&]+'
        found    = re.findall(pattern, raw_text)
        # Filter to .jpg and .png only (skip .sml thumbnails)
        for url in found:
            clean_url = url.split("?")[0]  # remove query params
            if clean_url.endswith(".jpg") or clean_url.endswith(".png"):
                if clean_url not in [f[1] for f in figures]:
                    figures.append(("", clean_url))

    return figures


# ── XML PARSER ────────────────────────────────────────────────────────────────

def parse_elsevier_xml(xml_bytes, doi):
    """
    Parse Elsevier full-text XML into structured clean text.
    Returns dict with 'text' and 'figures' keys.
    """
    try:
        root = etree.fromstring(xml_bytes)
    except Exception as e:
        print(f"  [XML PARSE ERROR] {e}")
        return None

    out = []

    # Title
    for tag in [f"{{{CE}}}title", f"{{{JA}}}title", f"{{{DC}}}title", ".//{*}title"]:
        node = root.find(tag) if tag.startswith("{") else root.find(tag)
        if node is not None:
            text = get_all_text(node)
            if text and len(text) > 5:
                out.append(f"TITLE: {text}\n")
                break

    # Authors -- collect surname + given-name pairs
    authors = []
    for author in root.findall(f".//{{{CE}}}author"):
        given  = author.find(f"{{{CE}}}given-name")
        family = author.find(f"{{{CE}}}surname")
        if family is not None:
            name = f"{given.text if given is not None else ''} {family.text or ''}".strip()
            if name:
                authors.append(name)
    # Fallback: dc:creator
    if not authors:
        for creator in root.findall(f".//{{{DC}}}creator"):
            if creator.text:
                authors.append(creator.text.strip())
    if authors:
        # Deduplicate while preserving order
        seen = set()
        unique_authors = []
        for a in authors:
            if a not in seen:
                seen.add(a)
                unique_authors.append(a)
        out.append(f"AUTHORS: {', '.join(unique_authors)}\n")

    # Abstract
    for tag in [f".//{{{CE}}}abstract", ".//{*}abstract"]:
        abstract = root.find(tag)
        if abstract is not None:
            abs_text = get_all_text(abstract)
            if abs_text and len(abs_text) > 50:
                out.append(f"\nABSTRACT:\n{abs_text}\n")
                break

    # Keywords
    keywords = []
    for kw in root.findall(f".//{{{CE}}}keyword"):
        kw_text = get_all_text(kw)
        if kw_text:
            keywords.append(kw_text)
    if keywords:
        out.append(f"\nKEYWORDS: {'; '.join(keywords)}\n")

    # Body -- walk sections
    body = root.find(f".//{{{CE}}}sections")
    if body is None:
        body = root.find(f".//{{{JA}}}body")
    if body is None:
        body = root.find(".//{*}body")

    if body is not None:
        _walk_body(body, out)
    else:
        # Fallback: extract all paragraphs directly
        for para in root.findall(f".//{{{CE}}}para"):
            text = get_all_text(para)
            if text and len(text) > 50:
                out.append(text)

    # Tables -- Elsevier stores tables in <floats> section OUTSIDE the body.
    # The body text only has references like "as shown in Table 2".
    # We must scan the entire document for all ce:table elements.
    all_tables = root.findall(f".//{{{CE}}}table")
    if not all_tables:
        all_tables = root.findall(".//{*}table")
    if all_tables:
        out.append("\n\nTABLES:")
        for table in all_tables:
            table_text = parse_ce_table(table)
            if table_text:
                out.append(f"\n{table_text}\n")

    # Extract figures
    figures = extract_figure_urls(root, xml_bytes)

    # Add figure count note at end
    if figures:
        out.append(f"\n[{len(figures)} FIGURES AVAILABLE -- downloaded separately to FIG_files/]")

    return {
        "text"   : "\n".join(out),
        "figures": figures,
    }


def _walk_body(node, out, depth=0):
    """
    Recursively walk body sections, extracting text, tables, and figure refs.
    """
    tag = etree.QName(node.tag).localname if node.tag else ""

    if tag == "section":
        # Section title
        label   = node.find(f"{{{CE}}}section-title")
        if label is None:
            label = node.find(f"{{{CE}}}label")
        if label is not None:
            title_text = get_all_text(label)
            if title_text:
                prefix = "\n\n" if depth == 0 else "\n"
                out.append(f"{prefix}{title_text.upper()}:")

        # Recurse into children
        for child in node:
            _walk_body(child, out, depth + 1)

    elif tag == "para":
        text = get_all_text(node)
        if text and len(text) > 20:
            out.append(text)

    elif tag == "table":
        table_text = parse_ce_table(node)
        if table_text:
            out.append(f"\n{table_text}\n")

    elif tag == "figure":
        # Note figure reference inline -- actual image downloaded separately
        caption_node = node.find(f".//{{{CE}}}caption")
        caption = get_all_text(caption_node) if caption_node is not None else ""
        label_node = node.find(f"{{{CE}}}label")
        label_text = get_all_text(label_node) if label_node is not None else "Figure"
        out.append(f"\n[{label_text}: {caption[:120]}]\n")

    else:
        # For any other container, recurse
        for child in node:
            _walk_body(child, out, depth)


# ── FIGURE DOWNLOADER ─────────────────────────────────────────────────────────

def download_figures(figures, doi, api_key):
    """
    Download all figure images for a paper using the Elsevier API key.
    Saves as doi_fig1.jpg, doi_fig2.jpg, etc. in FIG_files/.
    """
    if not figures:
        return 0

    saved = 0
    for i, (caption, url) in enumerate(figures, 1):
        # Determine extension from URL
        url_clean = url.split("?")[0]
        ext = ".png" if url_clean.endswith(".png") else ".jpg"
        fig_name = safe_filename(doi, f"_fig{i}{ext}")
        fig_path = os.path.join(FIG_OUTPUT, fig_name)

        if os.path.exists(fig_path):
            saved += 1
            continue

        try:
            r = requests.get(
                url,
                headers={"X-ELS-APIKey": api_key, "Accept": "*/*"},
                timeout=20
            )
            if r.status_code == 200 and len(r.content) > 500:
                with open(fig_path, "wb") as f:
                    f.write(r.content)
                saved += 1
            else:
                print(f"    [FIG {i}] Status {r.status_code} -- skipping")
        except Exception as e:
            print(f"    [FIG {i}] Error: {e}")
        time.sleep(0.2)

    return saved


# ── API FETCH ─────────────────────────────────────────────────────────────────

def fetch_elsevier_xml(doi, api_key):
    """Fetch full-text XML for one DOI. Returns (xml_bytes, status)."""
    url     = f"https://api.elsevier.com/content/article/doi/{doi}"
    headers = {"X-ELS-APIKey": api_key, "Accept": "text/xml"}

    try:
        r = requests.get(url, headers=headers, timeout=30)
        if r.status_code == 200:
            content = r.content
            # Check for body content indicating full text
            has_body = any(marker in content for marker in [
                b"<ce:sections>", b"<ce:section>", b"<body>", b"<ja:body>"
            ])
            return content, "full" if has_body else "abstract"
        elif r.status_code == 401:
            print(f"  [AUTH ERROR] Invalid API key")
            return None, "failed"
        elif r.status_code == 403:
            print(f"  [ACCESS DENIED] Check K-State network/VPN")
            return None, "failed"
        elif r.status_code == 404:
            print(f"  [NOT FOUND] {doi}")
            return None, "failed"
        else:
            print(f"  [ERROR] Status {r.status_code}")
            return None, "failed"
    except Exception as e:
        print(f"  [EXCEPTION] {e}")
        return None, "failed"


def process_doi(doi, api_key):
    """Fetch, parse, and save one DOI. Returns True if successful."""
    xml_bytes, status = fetch_elsevier_xml(doi, api_key)

    if status == "abstract":
        print(f"  [ABSTRACT ONLY] {doi} -- connect to K-State VPN and rerun")
        return False
    if status == "failed" or xml_bytes is None:
        return False

    parsed = parse_elsevier_xml(xml_bytes, doi)
    if not parsed or len(parsed["text"]) < 200:
        print(f"  [PARSE FAILED] Could not extract text for {doi}")
        return False

    word_count = len(parsed["text"].split())

    # Save text
    txt_path = os.path.join(TXT_OUTPUT, safe_filename(doi, ".txt"))
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(parsed["text"])

    # Download figures
    n_figs = download_figures(parsed["figures"], doi, api_key)

    print(f"  [OK] {doi} -- {word_count} words, {n_figs}/{len(parsed['figures'])} figures saved")
    return True


# ── MAIN ──────────────────────────────────────────────────────────────────────

def run():
    os.makedirs(TXT_OUTPUT, exist_ok=True)
    os.makedirs(FIG_OUTPUT, exist_ok=True)

    all_failed   = load_failed_dois(FAILED_LOG)
    elsevier     = [d for d in all_failed if is_elsevier(d)]
    non_elsevier = [d for d in all_failed if not is_elsevier(d)]

    print(f"\nElsevier DOIs to fetch : {len(elsevier)}")
    print(f"Non-Elsevier (skipped) : {len(non_elsevier)}")

    if not elsevier:
        print("No Elsevier DOIs found in failed list.")
        return

    # Test connectivity first
    print("\n=== Testing API key and network access ===")
    xml_bytes, status = fetch_elsevier_xml(elsevier[0], API_KEY)
    if status == "failed":
        print("[STOP] API key test failed.")
        return
    if status == "abstract":
        print("[STOP] Abstract only -- connect to K-State VPN and rerun.")
        return
    print("API key and network confirmed. Proceeding...\n")

    print("=== Fetching Elsevier DOIs ===")
    success_list, failed_list = [], []

    for doi in elsevier:
        if already_fetched(doi):
            print(f"  [SKIP] Already fetched: {doi}")
            success_list.append(doi)
            continue
        ok = process_doi(doi, API_KEY)
        (success_list if ok else failed_list).append(doi)
        time.sleep(0.5)

    # Save remaining failures
    still_failed = failed_list + non_elsevier
    with open(STILL_FAILED, "w") as f:
        f.write("\n".join(still_failed))

    print(f"\n{'='*60}")
    print(f"=== ELSEVIER API SUMMARY ===")
    print(f"{'='*60}")
    print(f"  Full text retrieved : {len(success_list)}")
    print(f"  Failed              : {len(failed_list)}")

    if non_elsevier:
        print(f"\n  Non-Elsevier DOIs needing manual action ({len(non_elsevier)}):")
        for doi in non_elsevier:
            print(f"    {doi}")

    if failed_list:
        print(f"\n  Elsevier DOIs still failed ({len(failed_list)}):")
        for doi in failed_list:
            print(f"    {doi}")

    print(f"\n  Text files  : {TXT_OUTPUT}/  (feeds into preprocessing notebook)")
    print(f"  Figure files: {FIG_OUTPUT}/  (send to vision model for data extraction)")
    print(f"  Failures    : {STILL_FAILED}")
    print(f"{'='*60}")


if __name__ == "__main__":
    run()

In [6]:
import os, glob
for f in glob.glob("Elsevier_TXT_files/*.txt"):
    os.remove(f)